# CS116 S6E10 - Model B2: RealMLP-TD Deep Learning Tabular Classifier (`B_realmlp_v1`)

## 1. Động lực & Giá trị Đa dạng hóa (Diversity Value):
- **Vấn đề của đội:** Hiện tại kho mô hình của cả nhóm và công khai (P247) có hơn 160 mô hình cây (LightGBM, XGBoost, CatBoost). Khi các mô hình cùng họ cây kết hợp với nhau, chúng có tương quan lỗi (error correlation) rất cao ($> 0.99$), khiến Meta-Stacker khó khai thác thêm thông tin mới.
- **Vì sao RealMLP là vũ khí bí mật của các Kaggle Grandmaster?**
  - **RealMLP-TD** (từ `pytabkit`, ICLR/NeurIPS Tabular Benchmark) là mạng nơ-ron nhiều tầng được thiết kế riêng cho dữ liệu bảng với cơ chế chuẩn hóa tham số (Weight Decay, LayerNorm, Mish/ReLU activations, Learning Rate Schedule tối ưu).
  - RealMLP không cắt cây theo trục tọa độ (axis-aligned splits) mà học các mặt phẳng phân cách mềm (smooth decision boundaries) trong không gian nhiều chiều.
  - Độ tương quan giữa RealMLP và GBDT chỉ khoảng $\rho \sim 0.97 - 0.98$. Trong pipeline Stacking v7 của nhóm, họ MLP luôn được cấp hệ số ưu tiên cao, giúp giải quyết các ca mà cây dự đoán sai.
- **Tuân thủ quy chuẩn sạch:**
  - Dùng đúng `data/folds.csv` chung của cả nhóm (seed 42).
  - Tách 10% nội bộ trong tập train làm tập dừng sớm (early stopping), **tuyệt đối không để rò rỉ vào fold validation đang chấm điểm**.

In [1]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from pytabkit import RealMLP_TD_Classifier

# Đảm bảo import được module trong src/
sys.path.insert(0, os.path.abspath("../src"))
from config import DATA_DIR, OOF_DIR, SUB_DIR, TARGET, ID, CAT_COLS, NUM_COLS, SCORE_COLS, FEATURES
from data import load_data
from utils import save_preds

warnings.filterwarnings("ignore")
MODEL_NAME = "B_realmlp_v1"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Huấn luyện mô hình: {MODEL_NAME} trên thiết bị: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Huấn luyện mô hình: B_realmlp_v1 trên thiết bị: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Nạp dữ liệu và Tiền xử lý cho Deep Learning

In [2]:
train, test = load_data()
y_true = train[TARGET].values
folds = train["fold"].values

def preprocess_for_realmlp(train_df, test_df):
    tr = train_df.copy()
    te = test_df.copy()
    
    # 1. Điền giá trị thiếu của Delay bằng giá trị tương ứng
    tr["Arrival Delay in Minutes"] = tr["Arrival Delay in Minutes"].fillna(tr["Departure Delay in Minutes"]).fillna(0)
    te["Arrival Delay in Minutes"] = te["Arrival Delay in Minutes"].fillna(te["Departure Delay in Minutes"]).fillna(0)
    
    # 2. Tạo một số đặc trưng tương tác phi tuyến mượt mà cho mạng nơ-ron
    for df in [tr, te]:
        arr = df["Arrival Delay in Minutes"]
        dep = df["Departure Delay in Minutes"]
        dist = df["Flight Distance"]
        
        df["Total_Delay"] = np.log1p(np.maximum(arr + dep, 0))
        df["Log_Distance"] = np.log1p(dist)
        df["Mean_Score"] = df[SCORE_COLS].mean(axis=1)
        df["Min_Score"] = df[SCORE_COLS].min(axis=1)
        df["Max_Score"] = df[SCORE_COLS].max(axis=1)
        df["Zero_Count"] = (df[SCORE_COLS] == 0).sum(axis=1).astype(float)
        
    # Chuyển các cột phân loại sang kiểu category để PyTabKit tự động tạo embeddings
    for c in CAT_COLS:
        tr[c] = tr[c].astype("category")
        te[c] = te[c].astype("category")
        
    cols_to_use = [c for c in tr.columns if c not in [ID, TARGET, "fold"]]
    return tr[cols_to_use], te[cols_to_use]

print("Chuẩn bị dữ liệu cho RealMLP...")
X_train, X_test = preprocess_for_realmlp(train, test)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print("Danh sách đặc trưng:", list(X_train.columns))

Chuẩn bị dữ liệu cho RealMLP...
X_train shape: (699635, 27), X_test shape: (299844, 27)
Danh sách đặc trưng: ['Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness', 'Departure Delay in Minutes', 'Arrival Delay in Minutes', 'Gender', 'Customer Type', 'Type of Travel', 'Class', 'Total_Delay', 'Log_Distance', 'Mean_Score', 'Min_Score', 'Max_Score', 'Zero_Count']


## 3. Huấn luyện RealMLP-TD qua 5 Fold chuẩn (Strict Clean OOF)

- Tuân thủ **Điều 3 trong STACKING.md**: Tách 10% trong tập train của fold làm tập dừng sớm, không dùng tập validation đang chấm điểm.
- Lưu lại xác suất OOF trên từng fold để đánh giá CV ROC-AUC thực chất.

In [3]:
N_TRAIN = len(train)
N_TEST = len(test)

oof_preds = np.zeros(N_TRAIN, dtype=np.float32)
test_preds = np.zeros(N_TEST, dtype=np.float32)
fold_aucs = []

print("=" * 65)
print(" BẮT ĐẦU HUẤN LUYỆN REALMLP-TD (5 FOLDS GPU)")
print("=" * 65)

for fold_k in range(5):
    t_start = time.time()
    va_idx = np.where(folds == fold_k)[0]
    tr_idx = np.where(folds != fold_k)[0]
    
    # Tách 10% từ nội bộ tr_idx để làm tập dừng sớm an toàn cho PyTabKit
    rng = np.random.default_rng(42 + fold_k)
    n_es = len(tr_idx) // 10
    perm = rng.permutation(len(tr_idx))
    es_sub_idx = tr_idx[perm[:n_es]]
    fit_sub_idx = tr_idx[perm[n_es:]]
    
    # Tạo tập train nội bộ ghép fit và es để truyền vào pytabkit với val_idxs
    X_tr_fold = pd.concat([X_train.iloc[fit_sub_idx], X_train.iloc[es_sub_idx]], axis=0)
    y_tr_fold = np.concatenate([y_true[fit_sub_idx], y_true[es_sub_idx]], axis=0)
    internal_val_idxs = np.arange(len(fit_sub_idx), len(X_tr_fold))
    
    # Khởi tạo mô hình RealMLP-TD tối ưu tốc độ trên GPU (batch_size 2048, 16 epochs)
    torch.set_float32_matmul_precision("high")
    clf = RealMLP_TD_Classifier(
        device=device,
        batch_size=2048,      # Tăng batch size lên 2048 (giảm 8 lần số bước)
        n_epochs=16,          # Giới hạn 16 epochs (tương đương TabM của teammate)
        random_state=42 + fold_k
    )
    
    # Fit mô hình với internal validation indices (không chạm vào fold_k)
    clf.fit(X_tr_fold, y_tr_fold, val_idxs=internal_val_idxs)
    
    # Dự đoán OOF sạch cho fold k
    pred_va = clf.predict_proba(X_train.iloc[va_idx])[:, 1]
    oof_preds[va_idx] = pred_va
    
    # Dự đoán test fold k
    pred_te = clf.predict_proba(X_test)[:, 1]
    test_preds += pred_te / 5.0
    
    auc_fold = roc_auc_score(y_true[va_idx], pred_va)
    fold_aucs.append(auc_fold)
    print(f"Fold {fold_k} [{time.time() - t_start:5.1f}s]: CV AUC = {auc_fold:.6f}")

overall_cv = roc_auc_score(y_true, oof_preds)
print("-" * 65)
print(f"★ TỔNG KẾT OVERALL CV AUC ({MODEL_NAME}): {overall_cv:.6f} ★")
print(f"  Mean Fold AUC: {np.mean(fold_aucs):.6f} ± {np.std(fold_aucs):.6f}")

 BẮT ĐẦU HUẤN LUYỆN REALMLP-TD (5 FOLDS GPU)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 4050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=16` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, w

Fold 0 [102.3s]: CV AUC = 0.957868


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=16` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and a

Fold 1 [104.9s]: CV AUC = 0.956635


...

Fold 2 [100.2s]: CV AUC = 0.958624


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=16` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and a

Fold 3 [ 99.3s]: CV AUC = 0.957741


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=16` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and a

Fold 4 [ 98.1s]: CV AUC = 0.957919
-----------------------------------------------------------------
★ TỔNG KẾT OVERALL CV AUC (B_realmlp_v1): 0.957695 ★
  Mean Fold AUC: 0.957757 ± 0.000640


## 4. Kiểm tra Độ Tương Quan Đa Dạng (Diversity Check với các Model Team)

In [4]:
from scipy.stats import rankdata, spearmanr

print("Kiểm tra tương quan rank giữa RealMLP và các model cây hiện có:")
for ref_name in ["quan_cat_dual_teacher_v4", "quan_xgb_dual_teacher_v4", "A_tabm_v1"]:
    path = OOF_DIR / f"oof_{ref_name}.npy"
    if path.exists():
        ref_oof = np.load(path)
        corr, _ = spearmanr(oof_preds, ref_oof)
        print(f"  -> Tương quan với {ref_name:<26}: rho = {corr:.5f}")

Kiểm tra tương quan rank giữa RealMLP và các model cây hiện có:
  -> Tương quan với quan_cat_dual_teacher_v4  : rho = 0.92969
  -> Tương quan với quan_xgb_dual_teacher_v4  : rho = 0.92679
  -> Tương quan với A_tabm_v1                 : rho = 0.90173


## 5. Lưu OOF & Test Predictions theo chuẩn CS116

In [5]:
# Lưu oof/oof_B_realmlp_v1.npy và oof/test_B_realmlp_v1.npy
save_preds(MODEL_NAME, oof_preds, test_preds, y_true)

# Xuất file submit độc lập
sub = pd.DataFrame({ID: test[ID], TARGET: test_preds})
sub_path = SUB_DIR / f"sub_{MODEL_NAME}.csv"
sub.to_csv(sub_path, index=False)
print(f"✓ Đã xuất file submit độc lập: {sub_path}")

Đã lưu oof_B_realmlp_v1.npy, test_B_realmlp_v1.npy | CV AUC = 0.957695
✓ Đã xuất file submit độc lập: D:\Luan\Study\CS116\cs116-kaggle-s6e10\submissions\sub_B_realmlp_v1.csv
